In [ ]:
from google.colab import files
uploaded = files.upload()  # ← 여기서 moe_code.zip 선택

Saving moe_code.zip to moe_code.zip


In [ ]:
# Cell 0: Setup
!pip install torch --index-url https://download.pytorch.org/whl/cu124 -q --upgrade
!pip install accelerate datasets tokenizers wandb -q

import torch
assert torch.cuda.is_available()
print(f"✅ {torch.cuda.get_device_name()} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys, os
SRC = "/content/drive/MyDrive/moe_project"
CKPT_DIR = f"{SRC}/checkpoints"
LOG_DIR = f"{SRC}/logs"
DATA_DIR = f"{SRC}/data"
TOKENIZER_DIR = f"{SRC}/tokenizer"

# ★ 이게 핵심 ★
sys.path.insert(0, f"{SRC}/code")

for d in [CKPT_DIR, LOG_DIR]: os.makedirs(d, exist_ok=True)

print("✅ Cell 0 완료")

✅ NVIDIA A100-SXM4-40GB / 42.4GB
Mounted at /content/drive
✅ Cell 0 완료


In [ ]:
import sys, os
SRC = "/content/drive/MyDrive/moe_project"
print("SRC exists:", os.path.exists(SRC))
print("code/ exists:", os.path.exists(f"{SRC}/code"))
print("model/ exists:", os.path.exists(f"{SRC}/code/model"))
print("moe_transformer.py:", os.path.exists(f"{SRC}/code/model/moe_transformer.py"))
print()
print("sys.path entries:")
for p in sys.path:
    if "drive" in p.lower() or "moe" in p.lower():
        print(f"  ✅ {p}")
    else:
        print(f"     {p}")

SRC exists: True
code/ exists: True
model/ exists: True
moe_transformer.py: True

sys.path entries:
  ✅ /content/drive/MyDrive/moe_project/code
     /content
     /env/python
     /usr/lib/python312.zip
     /usr/lib/python3.12
     /usr/lib/python3.12/lib-dynload
     
     /usr/local/lib/python3.12/dist-packages
     /usr/lib/python3/dist-packages
     /usr/local/lib/python3.12/dist-packages/IPython/extensions
     /root/.ipython


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive에 프로젝트 디렉토리 생성
import shutil
SRC = "/content/drive/MyDrive/moe_project"

# # 전체 코드 복사
# if os.path.exists(f"{SRC}/code"):
#     shutil.rmtree(f"{SRC}/code")
# shutil.copytree("/content/moe-project", f"{SRC}/code")

# # 하위 디렉토리 생성
# for d in ["checkpoints", "logs", "tokenizer", "data", "reports",
#           "logs/events", "logs/plots"]:
#     os.makedirs(f"{SRC}/{d}", exist_ok=True)

# print(f"✅ Code copied to {SRC}/code")
# print(f"✅ Directories ready at {SRC}")

Mounted at /content/drive


In [ ]:
# Cell 2: BPE 토크나이저 학습 (50K 문서 → vocab 32K)
# Colab 새 셀에 그대로 붙여넣고 실행

import sys, os

SRC = "/content/drive/MyDrive/moe_project"
sys.path.insert(0, f"{SRC}/code")

TOKENIZER_DIR = f"{SRC}/tokenizer"
os.makedirs(TOKENIZER_DIR, exist_ok=True)

from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

print("📦 Loading FineWeb-edu sample-10BT (50K docs)...")
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    "sample-10BT",
    split="train",
    streaming=True
)
texts = [ex["text"] for ex in dataset.take(50000)]
print(f"   Loaded {len(texts)} documents")

print("🔧 Training BPE tokenizer (vocab=32K)...")
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=32000,
    special_tokens=["<unk>", "<s>", "</s>", "<pad>"],
    min_frequency=2,
)
tokenizer.train_from_iterator(texts, trainer)
print("   ✅ BPE training complete!")

tokenizer.save(f"{TOKENIZER_DIR}/raw_bpe_tokenizer.json")
print(f"   ✅ Raw tokenizer saved")

from transformers import PreTrainedTokenizerFast
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/raw_bpe_tokenizer.json",
    unk_token="<unk>", pad_token="<pad>",
    bos_token="<s>", eos_token="</s>",
)
hf_tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"   ✅ HuggingFace format saved to {TOKENIZER_DIR}/")

test = hf_tokenizer("Hello, MoE Transformer!")
print(f"   ✅ Test: {test.tokens()} ({len(test)} tokens)")


📦 Loading FineWeb-edu sample-10BT (50K docs)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Cell 3 재실행 — block_size=1024 (기존 plan대로)
import sys, os, numpy as np
SRC = "/content/drive/MyDrive/moe_project"
sys.path.insert(0, f"{SRC}/code")
TOKENIZER_DIR = f"{SRC}/tokenizer"
DATA_DIR = f"{SRC}/data"

from transformers import PreTrainedTokenizerFast
from datasets import load_dataset

hf_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR,)

dataset = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT",
                       split="train", streaming=True)
all_ids = []
for i, ex in enumerate(dataset):
    if i >= 100000: break
    all_ids.extend(hf_tokenizer.encode(ex["text"]))
    if (i + 1) % 10000 == 0:
        print(f"  {i+1}/100000 ({len(all_ids):,} tokens)")

BLOCK_SIZE = 1024
num_chunks = len(all_ids) // BLOCK_SIZE
chunks = np.array(all_ids[:num_chunks * BLOCK_SIZE]).reshape(-1, BLOCK_SIZE)
split = int(chunks.shape[0] * 0.9)
np.save(f"{DATA_DIR}/train.npy", chunks[:split])
np.save(f"{DATA_DIR}/val.npy", chunks[split:])
print(f"Train: {chunks[:split].shape[0]} | Val: {chunks[split:].shape[0]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

  10000/100000 (10,384,137 tokens)
  20000/100000 (20,821,561 tokens)
  30000/100000 (32,167,167 tokens)
  40000/100000 (42,009,712 tokens)
  50000/100000 (52,934,586 tokens)
  60000/100000 (63,464,318 tokens)
  70000/100000 (74,061,521 tokens)
  80000/100000 (84,668,389 tokens)
  90000/100000 (94,441,342 tokens)
  100000/100000 (105,015,556 tokens)
Train: 92298 | Val: 10256


In [ ]:
# Step 4. 실제 모델 학습
import sys, os, time, math, json, glob
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from accelerate import Accelerator

SRC = "/content/drive/MyDrive/moe_project"
sys.path.insert(0, f"{SRC}/code")

CKPT_DIR, LOG_DIR, DATA_DIR = f"{SRC}/checkpoints", f"{SRC}/logs", f"{SRC}/data"
for d in [CKPT_DIR, LOG_DIR]: os.makedirs(d, exist_ok=True)

from model.moe_transformer import MoETransformer

accelerator = Accelerator(mixed_precision="bf16")
device = accelerator.device
print(f"Device: {device}")

train_data = torch.from_numpy(np.load(f"{DATA_DIR}/train.npy")).long()
train_loader = DataLoader(TensorDataset(train_data), batch_size=4, shuffle=True)

model = MoETransformer(vocab_size=32000)
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# [수정 ①] Gradient Accumulation으로 batch=4→16 효과
GRAD_ACCUM = 4

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
# [수정 ②] LR 재설정 — Cosine 새 주기 시작
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5000)
model, optimizer, train_loader = accelerator.prepare(model, optimizer, train_loader)

def load_latest():
    ckpts = glob.glob(f"{CKPT_DIR}/moe_*.pt")
    if not ckpts: return 0
    latest = max(ckpts, key=os.path.getmtime)
    ckpt = torch.load(latest, map_location="cpu")
    accelerator.unwrap_model(model).load_state_dict(ckpt["model_state_dict"])
    # optimizer/scheduler는 새로 시작 (LR 재설정)
    print(f"Loaded checkpoint: step {ckpt['step']}, loss {ckpt['loss']:.4f}")
    return 0  # 처음부터 다시 학습 (LR 재설정했으므로)

def save_ckpt(step, loss):
    unwrapped = accelerator.unwrap_model(model)
    path = f"{CKPT_DIR}/moe_v2_{step}.pt"
    torch.save({"step": step, "loss": loss,
        "model_state_dict": unwrapped.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict()}, path)
    print(f"Checkpoint: step {step}")

start_step = load_latest()
data_iter, MAX_STEPS, SEQ_LEN = iter(train_loader), 5000, 1024
log_file = f"{LOG_DIR}/metrics_r002.jsonl"

for step in range(start_step, MAX_STEPS):
    t0 = time.time()
    optimizer.zero_grad()
    total_main_loss = 0.0

    for micro in range(GRAD_ACCUM):
        try: batch = next(data_iter)
        except StopIteration: data_iter, batch = iter(train_loader), next(iter(train_loader))

        input_ids = batch[0][:, :SEQ_LEN].to(device)
        logits, loss, main_loss, aux_loss, z_loss = model(input_ids, labels=input_ids)
        loss = loss / GRAD_ACCUM
        accelerator.backward(loss)
        total_main_loss += main_loss.item()

    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step(); scheduler.step()

    if step % 100 == 0:
        avg_loss = total_main_loss / GRAD_ACCUM
        ppl = math.exp(min(avg_loss, 20))
        print(f"Step {step:5d} | Loss {avg_loss:.4f} | Aux {aux_loss.item():.4f} | PPL {ppl:.2f} | {time.time()-t0:.1f}s")
        with open(log_file, "a") as f:
            f.write(json.dumps({"step": step, "loss": avg_loss,
                "aux_loss": aux_loss.item(), "z_loss": z_loss.item(), "ppl": ppl, "grad_norm": grad_norm.item()})+"\n")
    if step > 0 and step % 1000 == 0: save_ckpt(step, avg_loss)

save_ckpt(MAX_STEPS, avg_loss)

KeyboardInterrupt: 

In [ ]:
# 모델 학습 수준 확인
import sys, os, glob, torch

SRC = "/content/drive/MyDrive/moe_project"
sys.path.insert(0, f"{SRC}/code")
CKPT_DIR = f"{SRC}/checkpoints"
TOKENIZER_DIR=f"{SRC}/tokenizer"

from model.moe_transformer import MoETransformer
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)
model = MoETransformer(vocab_size=32000)
model.eval()

ckpts = glob.glob(f"{CKPT_DIR}/moe_*.pt")
if ckpts:
    latest = max(ckpts, key=os.path.getmtime)
    ckpt = torch.load(latest, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    status = f"TRAINED (step {ckpt['step']}, loss {ckpt['loss']:.4f})"
else:
    status = "UNTRAINED (random weights)"

print(f"Status: {status}\n")

def generate(model, tokenizer, prompt, max_new_tokens=50):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, *_ = model(input_ids)   # ← 5개 반환값 중 logits만 사용
            next_id = logits[0, -1, :].argmax().unsqueeze(0).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_id], dim=1)
            if next_id.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

prompts = ["The future of AI is", "Machine learning models", "Once upon a time", "AI는 현재 우리 삶을 어떻게"]

print("=" * 60)
print(f"[{status}]")
print("=" * 60)
for prompt in prompts:
    output = generate(model, tokenizer, prompt, 50)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {output}")
    print("-" * 40)

Status: TRAINED (step 5000, loss 3.8991)

[TRAINED (step 5000, loss 3.8991)]

Prompt: The future of AI is
Output: The future of AI is a very important aspect of the future of AI.
The future of AI is a very important aspect of AI. AI is a very important aspect of AI. AI is a very important aspect of AI. AI is a very important aspect of AI.
----------------------------------------

Prompt: Machine learning models
Output: Machine learning models.
- The first step is to create a model that is used to model the model.
- The model is used to model the model.
- The model is used to model the model.
- The model is used to model the
----------------------------------------

Prompt: Once upon a time
Output: Once upon a time, the first thing that the author of the book was the author’s name.
The author’s name was changed to the author’s name, and the author’s name was changed to the author’s name.
The author
----------------------------------------

Prompt: AI는 현재 우리 삶을 어떻게
Output: AI는 현재 우리 삶을 어